# Video Segmentation with Multiple Bounding Boxes in SAM3.1

This notebook demonstrates how to use bounding boxes for video segmentation in SAM3.1 (Object Multiplex).

**Main Use Case**: Track multiple objects (2+) with their own bounding boxes and propagate across the entire video.

SAM 3.1 uses an Object Multiplex approach which is more efficient than SAM 3, grouping objects into fixed-capacity buckets to reduce redundant computation.

## Setup

In [1]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import os
import glob

device = torch.device("cuda")

if device.type == "cuda":
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

In [2]:
import sam3

from sam3.model_builder import build_sam3_multiplex_video_predictor
from sam3.visualization_utils import (
    load_frame,
    prepare_masks_for_visualization,
    visualize_formatted_frame_output,
    show_box,
    show_mask,
)

# Build SAM 3.1 model (multiplex version)
predictor = build_sam3_multiplex_video_predictor()

# Get path to video
sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")
video_path = f"{sam3_root}/assets/videos/bedroom.mp4"

print(f"Loading video from: {video_path}")

/cluster/scratch/niacobone/sam3/myenv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


dynamic_multimask_via_stability is reset to False in the multiplex model


INFO 2026-05-22 19:37:54,653 3751568 sam3_multiplex_base.py: 336: `setting max_num_objects` to 16 -- creating num_obj_for_compile=1 objects for torch.compile cache


Missing keys (64): ['detector.backbone.vision_backbone.trunk.blocks.0.attn.freqs_cis_real', 'detector.backbone.vision_backbone.trunk.blocks.0.attn.freqs_cis_imag', 'detector.backbone.vision_backbone.trunk.blocks.1.attn.freqs_cis_real', 'detector.backbone.vision_backbone.trunk.blocks.1.attn.freqs_cis_imag', 'detector.backbone.vision_backbone.trunk.blocks.2.attn.freqs_cis_real', 'detector.backbone.vision_backbone.trunk.blocks.2.attn.freqs_cis_imag', 'detector.backbone.vision_backbone.trunk.blocks.3.attn.freqs_cis_real', 'detector.backbone.vision_backbone.trunk.blocks.3.attn.freqs_cis_imag', 'detector.backbone.vision_backbone.trunk.blocks.4.attn.freqs_cis_real', 'detector.backbone.vision_backbone.trunk.blocks.4.attn.freqs_cis_imag']...
Loading video from: /cluster/scratch/niacobone/sam3/sam3/../assets/videos/bedroom.mp4


In [3]:
# Load video frames for visualization
cap = cv2.VideoCapture(video_path)
video_frames_for_vis = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    video_frames_for_vis.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()

frame0 = video_frames_for_vis[0]
width, height = frame0.shape[1], frame0.shape[0]
print(f"Video resolution: {width}x{height}, Total frames: {len(video_frames_for_vis)}")

Video resolution: 960x540, Total frames: 200


## Track 2 Objects with Bounding Boxes and Propagate Through Video

This is the primary way to use multiple bounding boxes in SAM3.1 - provide one box per object and track them across the entire video.

In [4]:
# Initialize inference session
response = predictor.handle_request(
    request=dict(
        type="start_session",
        resource_path=video_path,
    )
)
session_id = response["session_id"]

print(f"Session initialized. Session ID: {session_id}")
print("Ready for prompts.")

frame loading (OpenCV) [rank=0]: 100%|██████████| 200/200 [00:00<00:00, 865.78it/s]
INFO 2026-05-22 19:37:58,780 3751568 sam3_base_predictor.py: 153: started new session a23f53de-ae40-4a67-8f7b-82258079ac50


Session initialized. Session ID: a23f53de-ae40-4a67-8f7b-82258079ac50
Ready for prompts.


In [5]:
# Helper function to propagate in video
def propagate_in_video(predictor, session_id):
    outputs_per_frame = {}
    for response in predictor.handle_stream_request(
        request=dict(
            type="propagate_in_video",
            session_id=session_id,
        )
    ):
        outputs_per_frame[response["frame_index"]] = response["outputs"]
    return outputs_per_frame

# Helper function to convert pixel coordinates to relative coordinates
def abs_to_rel_coords(coords, img_width, img_height, coord_type="box"):
    """Convert absolute coordinates to relative coordinates (0-1 range)
    
    Args:
        coords: For box: [x_min, y_min, x_max, y_max] (pixel coordinates)
        coord_type: 'box' for [x_min, y_min, x_max, y_max] format
    """
    if coord_type == "box":
        # Convert from [xmin, ymin, xmax, ymax] to [x, y, w, h] relative format
        x_min, y_min, x_max, y_max = coords
        x_rel = x_min / img_width
        y_rel = y_min / img_height
        w_rel = (x_max - x_min) / img_width
        h_rel = (y_max - y_min) / img_height
        return [x_rel, y_rel, w_rel, h_rel]
    else:
        raise ValueError(f"Unknown coord_type: {coord_type}")

In [6]:
# Step 1: Define bounding boxes for two objects on frame 0
ann_frame_idx = 0  # Initial frame where we provide prompts

# Object 1: Child on the right (object ID 1)
obj_id_1 = 1
box_1_pixel = np.array([300, 0, 500, 400], dtype=np.float32)  # [x_min, y_min, x_max, y_max]
box_1_rel = abs_to_rel_coords(box_1_pixel, width, height, coord_type="box")

# Object 2: Child on the left (object ID 2)
obj_id_2 = 2
box_2_pixel = np.array([100, 200, 250, 500], dtype=np.float32)  # [x_min, y_min, x_max, y_max]
box_2_rel = abs_to_rel_coords(box_2_pixel, width, height, coord_type="box")

print(f"Object 1 (ID={obj_id_1}): Box in pixels (xmin,ymin,xmax,ymax): {box_1_pixel}")
print(f"Object 1: Box in relative coords (x,y,w,h): {box_1_rel}")
print(f"Object 2 (ID={obj_id_2}): Box in pixels (xmin,ymin,xmax,ymax): {box_2_pixel}")
print(f"Object 2: Box in relative coords (x,y,w,h): {box_2_rel}")

Object 1 (ID=1): Box in pixels (xmin,ymin,xmax,ymax): [300.   0. 500. 400.]
Object 1: Box in relative coords (x,y,w,h): [0.3125, 0.0, 0.20833333333333334, 0.7407407407407407]
Object 2 (ID=2): Box in pixels (xmin,ymin,xmax,ymax): [100. 200. 250. 500.]
Object 2: Box in relative coords (x,y,w,h): [0.10416666666666667, 0.37037037037037035, 0.15625, 0.5555555555555556]


In [7]:
# Step 2: Add box prompt for object 1
print(f"Adding box prompt for object {obj_id_1} on frame {ann_frame_idx}...")

response_1 = predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=ann_frame_idx,
        obj_id=obj_id_1,
        bounding_boxes=torch.tensor([box_1_rel], dtype=torch.float32),
        bounding_box_labels=torch.tensor([1], dtype=torch.int32),  # 1 = foreground
    )
)
out_1 = response_1["outputs"]
print(f"Object {obj_id_1} prompt added. Output object IDs: {out_1.get('out_obj_ids', [])}")

INFO 2026-05-22 19:37:58,807 3751568 sam3_multiplex_tracking.py:1677: Running add_prompt on frame 0


Adding box prompt for object 1 on frame 0...


ModuleNotFoundError: No module named 'flash_attn_interface'

In [ ]:
# Step 3: Add box prompt for object 2
print(f"Adding box prompt for object {obj_id_2} on frame {ann_frame_idx}...")

response_2 = predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=ann_frame_idx,
        obj_id=obj_id_2,
        bounding_boxes=torch.tensor([box_2_rel], dtype=torch.float32),
        bounding_box_labels=torch.tensor([1], dtype=torch.int32),  # 1 = foreground
    )
)
out_2 = response_2["outputs"]
print(f"Object {obj_id_2} prompt added. Output object IDs: {out_2.get('out_obj_ids', [])}")

In [ ]:
# Step 4: Visualize the prompts and initial segmentations on frame 0
plt.figure(figsize=(12, 8))
plt.imshow(video_frames_for_vis[ann_frame_idx])

# Draw boxes
show_box(box_1_pixel, plt.gca())
show_box(box_2_pixel, plt.gca())

# Draw segmentation masks for both objects
out_obj_ids = out_2["out_obj_ids"]
out_binary_masks = out_2["out_binary_masks"]

for i, obj_id in enumerate(out_obj_ids):
    if i < len(out_binary_masks):
        show_mask(out_binary_masks[i], plt.gca(), obj_id=obj_id)

plt.title(f"Frame {ann_frame_idx}: Initial Segmentation with 2 Bounding Box Prompts")
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Object {obj_id_1}: Segmentation mask displayed (green)")
print(f"Object {obj_id_2}: Segmentation mask displayed (orange)")

In [ ]:
# Step 5: Propagate prompts through the entire video
print("\nPropagating prompts through the entire video...")
print("This tracks both objects across all frames.\n")

outputs_per_frame = propagate_in_video(predictor, session_id)

# Reformat for visualization
outputs_per_frame_formatted = prepare_masks_for_visualization(outputs_per_frame)

print(f"✓ Propagation complete. Segmented {len(outputs_per_frame)} frames.")

In [ ]:
# Step 6: Visualize tracking results every N frames
vis_frame_stride = 30
frame_indices = list(range(0, len(video_frames_for_vis), vis_frame_stride))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for plot_idx, frame_idx in enumerate(frame_indices):
    ax = axes[plot_idx]
    ax.imshow(video_frames_for_vis[frame_idx])
    
    # Draw segmentation masks for this frame using formatted output
    if frame_idx in outputs_per_frame_formatted:
        frame_output = outputs_per_frame_formatted[frame_idx]
        if isinstance(frame_output, dict):
            for out_obj_id, out_mask in frame_output.items():
                show_mask(out_mask, ax, obj_id=out_obj_id)
        else:
            # Handle alternative output format
            out_binary_masks = frame_output.get("out_binary_masks", [])
            out_obj_ids = frame_output.get("out_obj_ids", [])
            for i, obj_id in enumerate(out_obj_ids):
                if i < len(out_binary_masks):
                    show_mask(out_binary_masks[i], ax, obj_id=obj_id)
    
    ax.set_title(f"Frame {frame_idx}")
    ax.axis('off')

plt.suptitle("SAM3.1 Multi-Object Video Segmentation with Bounding Box Prompts", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nShowing frames: {frame_indices}")
print(f"Stride: {vis_frame_stride} frames")

In [ ]:
# Step 7: Summary statistics
print("\n" + "="*60)
print("TRACKING SUMMARY")
print("="*60)

for frame_idx, frame_data in outputs_per_frame.items():
    if frame_idx == 0:  # Show details for frame 0
        out_obj_ids = frame_data.get("out_obj_ids", [])
        out_binary_masks = frame_data.get("out_binary_masks", [])
        print(f"Frame {frame_idx}: Tracking {len(out_obj_ids)} objects")
        for i, obj_id in enumerate(out_obj_ids):
            if i < len(out_binary_masks):
                mask_size = np.sum(out_binary_masks[i])
                print(f"  Object {obj_id}: mask size = {mask_size:.0f} pixels")
        break

# Calculate per-object statistics
obj_stats = {}
for frame_idx, frame_data in outputs_per_frame.items():
    out_obj_ids = frame_data.get("out_obj_ids", [])
    out_binary_masks = frame_data.get("out_binary_masks", [])
    for i, obj_id in enumerate(out_obj_ids):
        if obj_id not in obj_stats:
            obj_stats[obj_id] = {"frames": 0, "mask_sizes": []}
        obj_stats[obj_id]["frames"] += 1
        if i < len(out_binary_masks):
            obj_stats[obj_id]["mask_sizes"].append(np.sum(out_binary_masks[i]))

print("\n")
for obj_id, stats in sorted(obj_stats.items()):
    print(f"Object {obj_id}:")
    print(f"  - Tracked in {stats['frames']}/{len(outputs_per_frame)} frames")
    if stats["mask_sizes"]:
        avg_size = np.mean(stats["mask_sizes"])
        print(f"  - Average mask size: {avg_size:.0f} pixels")

print("\nTracking completed successfully!")
print("="*60)

## API Reference: Using Multiple Bounding Boxes in SAM 3.1

### Key Function: `handle_request()` with `add_prompt`

```python
response = predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=0,
        obj_id=1,
        bounding_boxes=torch.tensor([[x, y, w, h]], dtype=torch.float32),
    )
)
```

### Workflow: 3 Steps

1. **Initialize Session**
   ```python
   response = predictor.handle_request(
       request=dict(
           type="start_session",
           resource_path=video_path,
       )
   )
   session_id = response["session_id"]
   ```

2. **Add Box Prompts for Each Object**
   ```python
   # Object 1
   response = predictor.handle_request(
       request=dict(
           type="add_prompt",
           session_id=session_id,
           frame_index=0,
           obj_id=1,
           bounding_boxes=torch.tensor([box_1_rel], dtype=torch.float32),
       )
   )
   
   # Object 2
   response = predictor.handle_request(
       request=dict(
           type="add_prompt",
           session_id=session_id,
           frame_index=0,
           obj_id=2,
           bounding_boxes=torch.tensor([box_2_rel], dtype=torch.float32),
       )
   )
   ```

3. **Propagate Through Video**
   ```python
   outputs_per_frame = {}
   for response in predictor.handle_stream_request(
       request=dict(
           type="propagate_in_video",
           session_id=session_id,
       )
   ):
       outputs_per_frame[response["frame_index"]] = response["outputs"]
   ```

### Important Notes

- **Session-based**: SAM 3.1 uses sessions instead of inference states
- **One box per add_prompt call**: Each call adds one box to one object
- **Unique IDs**: Each object must have a unique `obj_id`
- **Normalized coordinates**: Box must be in range [0, 1] in format [x, y, w, h]
  - x, y: top-left corner (relative)
  - w, h: width and height (relative)
  - Convert from pixels: `x = x_min/width, y = y_min/height, w = (x_max-x_min)/width, h = (y_max-y_min)/height`
- **Object Multiplex**: SAM 3.1 groups objects into fixed-capacity buckets for efficiency
- **Output format**: Returns `out_obj_ids`, `out_binary_masks`, and optionally `out_boxes_xywh`

### Closing a Session

```python
predictor.handle_request(
    request=dict(
        type="close_session",
        session_id=session_id,
    )
)
```

## Optional: Comparing Multiple Box Proposals for One Object

In [ ]:
# Reset session for comparing multiple box proposals
_ = predictor.handle_request(
    request=dict(
        type="reset_session",
        session_id=session_id,
    )
)

ann_frame_idx = 0

# Multiple proposals for the same object (from different detectors/strategies)
box_proposals = [
    {"proposal_id": 1, "box": [290, 20, 510, 420], "label": "Loose"},
    {"proposal_id": 2, "box": [310, 50, 480, 380], "label": "Tight"},
    {"proposal_id": 3, "box": [300, 0, 500, 400], "label": "Medium"},
]

proposal_masks = {}

for proposal in box_proposals:
    proposal_id = proposal["proposal_id"]
    box_pixel = np.array(proposal["box"], dtype=np.float32)
    box_rel = abs_to_rel_coords(box_pixel, width, height, coord_type="box")

    response = predictor.handle_request(
        request=dict(
            type="add_prompt",
            session_id=session_id,
            frame_index=ann_frame_idx,
            obj_id=100 + proposal_id,  # Use 100+ IDs to avoid conflict with earlier objects
            bounding_boxes=torch.tensor([box_rel], dtype=torch.float32),
            bounding_box_labels=torch.tensor([1], dtype=torch.int32),  # 1 = foreground
        )
    )
    
    outputs = response["outputs"]
    out_binary_masks = outputs.get("out_binary_masks", [])
    
    if len(out_binary_masks) > 0:
        proposal_masks[proposal_id] = {
            "mask": out_binary_masks[0],
            "label": proposal["label"],
            "box": box_pixel,
        }

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, (proposal_id, data) in enumerate(proposal_masks.items()):
    ax = axes[idx]
    ax.imshow(video_frames_for_vis[ann_frame_idx])
    show_box(data["box"], ax)
    show_mask(data["mask"], ax, obj_id=100 + proposal_id)
    ax.set_title(f"Proposal {proposal_id}: {data['label']}")
    ax.axis('off')

plt.suptitle("Comparing Box Proposals (Select Best for Propagation)", fontsize=14)
plt.tight_layout()
plt.show()

## Close Session

Always close the session when done to free up resources.

In [ ]:
# Close the session to free resources
_ = predictor.handle_request(
    request=dict(
        type="close_session",
        session_id=session_id,
    )
)

print("Session closed successfully.")